# Explore HiggsLab events and a measurement

This notebook is an optional tour after completing the [quickstart](../docs/setup.md). It reads the starter's public dataset and, if you ran the baseline, displays the result file. The reproducible command-line pipeline remains the source of truth for a submission.

> **Provenance:** the current data are a synthetic teaching surrogate, not ATLAS data or detector simulation. See [data and provenance](../docs/data.md).

Run the notebook in order. No outputs or saved state are committed, so your results are always produced in your current environment.

In [ ]:
from pathlib import Path
import json
import numpy as np

HERE = Path.cwd()
ROOT = next((candidate for candidate in (HERE, *HERE.parents) if (candidate / 'pyproject.toml').exists()), HERE)
DATASET = ROOT / 'datasets' / 'public'
print('Dataset directory exists:', DATASET.exists())
print('Data files:', [str(p.relative_to(ROOT)) for p in DATASET.rglob('*') if p.is_file()] if DATASET.exists() else 'Generate/copy the public sample for this checkout')

## Read the sample metadata

Metadata records schema, units, teaching origin, process labels, and partition meanings. Always inspect it before interpreting the arrays.

In [ ]:
meta_path = DATASET / 'mc' / 'pool_meta.json'
if meta_path.exists():
    metadata = json.loads(meta_path.read_text(encoding='utf-8'))
    print(json.dumps(metadata, indent=2))
else:
    print('No pool_meta.json in this checkout yet.')

## Inspect columns without using metadata as model input

The labeled MC pool supports training and template construction. Labels and weights are useful for analysis bookkeeping; event IDs and partition IDs are bookkeeping too. Do not pass these fields into a classifier as physics features.

In [ ]:
pool_path = DATASET / 'mc' / 'pool.npz'
if pool_path.exists():
    with np.load(pool_path, allow_pickle=False) as pool:
        print('Columns:', sorted(pool.files))
        print('Rows:', len(pool['label']))
        print('Label counts:', {int(label): int((pool['label'] == label).sum()) for label in np.unique(pool['label'])})
        print('Partition counts:', {int(part): int((pool['partition'] == part).sum()) for part in np.unique(pool['partition'])})
        print('Total signed weights by label:', {int(label): float(pool['weight'][pool['label'] == label].sum()) for label in np.unique(pool['label'])})
else:
    print('No MC pool found. Follow the quickstart for this checkout.')

## Inspect the baseline result

Run `uv run higgslab run datasets/public output` from the repository root first. The result contract contains a parameter of interest named `mu` and a positive uncertainty.

In [ ]:
result_path = ROOT / 'output' / 'RESULT.json'
if result_path.exists():
    result = json.loads(result_path.read_text(encoding='utf-8'))
    print(json.dumps(result, indent=2))
else:
    print('No result yet. From the repository root run: uv run higgslab run datasets/public output')

## Explore one question

Choose a feature or fit exercise and predict what it will change before editing code. Record the baseline result, make one change, rerun `uv run higgslab check`, and compare the same generated suite. See the [analysis map](../docs/analysis.md), [classifier path](../docs/classifier.md), and [fit exercises](../docs/fit.md). A classifier metric is a diagnostic; the shared outcome is μ recovery and uncertainty calibration.